In [19]:
import os
from dotenv import load_dotenv
from github import Github
import json


import warnings
warnings.filterwarnings("ignore")


In [14]:
load_dotenv()  

TOKEN = os.getenv("GITHUB_TOKEN")
g = Github(TOKEN)

In [15]:
def safe_call(fn, *args, **kwargs):
    """Call a PyGithub function, retrying automatically on rate limit errors."""
    while True:
        try:
            return fn(*args, **kwargs)
        except RateLimitExceededException:
            reset_time = g.get_rate_limit().core.reset.timestamp()
            sleep_for = max(reset_time - time.time(), 1) + 5
            print(f"Rate limit hit. Sleeping for {sleep_for:.0f} seconds...")
            time.sleep(sleep_for)
        except GithubException as e:
            print(f"GitHub API error: {e}")
            raise


In [30]:
def collect_pr_data(repo, state="closed", max_prs=50):
    """
    Collect GitHub Pull Request data for training a PR Review LLM.
    """
    results = []
    pulls = safe_call(
        repo.get_pulls,
        state=state,
        sort="updated",
        direction="desc"
    )

    for idx, pr in enumerate(pulls):
        if idx >= max_prs:
            break
        print(f"[{idx+1}/{max_prs}] PR #{pr.number}: {pr.title}")

        pr_record = {
            "repository": repo.full_name,
            "repository_url": repo.html_url,
            "repository_language": repo.language,

            "pr_number": pr.number,
            "title": pr.title,
            "description": pr.body,

            "url": pr.html_url,

            "state": pr.state,
            "merged": pr.merged,

            "author": pr.user.login if pr.user else None,

            "created_at":
                pr.created_at.isoformat()
                if pr.created_at else None,

            "updated_at":
                pr.updated_at.isoformat()
                if pr.updated_at else None,

            "closed_at":
                pr.closed_at.isoformat()
                if pr.closed_at else None,

            "base_branch": pr.base.ref,
            "head_branch": pr.head.ref,

            "base_sha": pr.base.sha,
            "head_sha": pr.head.sha,

            "merge_commit_sha": pr.merge_commit_sha,

            "changed_files": pr.changed_files,
            "commits": pr.commits,

            "additions": pr.additions,
            "deletions": pr.deletions,

            "labels": [
                label.name
                for label in pr.labels
            ],

            "files": [],
            "discussion_comments": [],
            "review_comments": [],
            "reviews": [],
            "commits_data": []
        }

        for file in safe_call(pr.get_files):

            if file.patch is None:
                continue

            pr_record["files"].append({

                "filename": file.filename,

                "status": file.status,

                "additions": file.additions,
                "deletions": file.deletions,
                "changes": file.changes,

                "patch": file.patch,

                "blob_url": file.blob_url,
                "raw_url": file.raw_url

            })

        for commit in safe_call(pr.get_commits):

            pr_record["commits_data"].append({

                "sha": commit.sha,
                "message": commit.commit.message,
                "author":
                    commit.author.login
                    if commit.author else None,
                "date":
                    commit.commit.author.date.isoformat()
                    if commit.commit.author else None
            })

        for comment in safe_call(pr.get_issue_comments):

            pr_record["discussion_comments"].append({
                "user":
                    comment.user.login
                    if comment.user else None,
                "body": comment.body,
                "created_at":
                    comment.created_at.isoformat()
                    if comment.created_at else None,
                "url": comment.html_url
            })

        for comment in safe_call(pr.get_review_comments):
            pr_record["review_comments"].append({
                "user":
                    comment.user.login
                    if comment.user else None,
                "body": comment.body,
                "path": comment.path,
                "line": comment.line,
                "commit_id": comment.commit_id,
                "created_at":
                    comment.created_at.isoformat()
                    if comment.created_at else None,
                "url": comment.html_url
            })

        for review in safe_call(pr.get_reviews):
            pr_record["reviews"].append({
                "user":
                    review.user.login
                    if review.user else None,

                "state": review.state,
                "body": review.body,
                "submitted_at":
                    review.submitted_at.isoformat()
                    if review.submitted_at else None

            })

        pr_record["statistics"] = {
            "num_files":
                len(pr_record["files"]),
            "num_discussion_comments":
                len(pr_record["discussion_comments"]),
            "num_review_comments":
                len(pr_record["review_comments"]),
            "num_reviews":
                len(pr_record["reviews"]),
            "num_commits":
                len(pr_record["commits_data"])
        }
        results.append(pr_record)

    return results

In [27]:
data = collect_pr_data(repo, state="closed", max_prs=50)
print(f"Collected data for {len(data)} pull requests.")


[1/50] PR #21709: Fixed #37236 -- Allowed altering spatial indexes on RasterField.
[2/50] PR #21629: Fixed #27734 -- Made parallel test workers reuse database clones of exited workers.
[3/50] PR #21706: Removed advice to include ticket numbers in tests.
[4/50] PR #21693: Fixed #37238 -- Fixed unintentional fallback to python default for a db_default.
[5/50] PR #18506: Fixes [#25656] Stop displaying links in recent changes when user does not have view permission
[6/50] PR #21699: Fixed #37235 -- Added compatibility for sqlparse 0.5.5.
[7/50] PR #21700: Fixed #37240 -- Fixed simplify_regex() with multiple unnamed groups.
[8/50] PR #21704: Updated asgiref dependency in free-threaded requirements.
[9/50] PR #21703: Fixed #31923 -- Added Cross-Origin-Embedder-Policy and Cross-Origin-Resource-Policy header support.
[10/50] PR #21701: Fixed #31923 -- Added Cross-Origin-Embedder-Policy and Cross-Origin-Resource-Policy header support.
[11/50] PR #21702: Fixed #31923 -- Added Cross-Origin-Embedd

In [31]:
output_dir = "data"
os.makedirs(output_dir, exist_ok=True)

repo_slug = repo.full_name.replace("/", "_")
output_path = os.path.join(output_dir, f"{repo_slug}_pr_data.jsonl")

with open(output_path, "w", encoding="utf-8") as f:
    for record in data:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Saved {len(data)} PRs to {output_path}")

Saved 50 PRs to data\django_django_pr_data.jsonl


In [29]:
if data:
    sample = data[0]
    print("Sample PR:", sample["title"])
    print("Discussion comments:", len(sample["discussion_comments"]))
    print("Review comments:", len(sample["review_comments"]))
    print("Reviews:", len(sample["reviews"]))


Sample PR: Fixed #37236 -- Allowed altering spatial indexes on RasterField.
Discussion comments: 2
Review comments: 0
Reviews: 1


In [ ]:
import os
import json

# ==========================================================
# Repositories to Scrape
# ==========================================================

repositories = [

    # Python
    "django/django",
    "fastapi/fastapi",
    "pallets/flask",
    "pydantic/pydantic",

    # Java
    "spring-projects/spring-framework",
    "apache/kafka",

    # JavaScript / TypeScript
    "microsoft/vscode",
    "facebook/react",
    "vercel/next.js",

    # Go
    "kubernetes/kubernetes",
    "gin-gonic/gin",

    # C++
    "opencv/opencv",

    # Machine Learning
    "tensorflow/tensorflow",
    "pytorch/pytorch",

    # Database / SQL
    "postgres/postgres",
    "mysql/mysql-server"
]

# ==========================================================
# Output Directory
# ==========================================================

output_dir = "data"
os.makedirs(output_dir, exist_ok=True)

total_prs = 0

# ==========================================================
# Collect Dataset
# ==========================================================

for repo_name in repositories:

    print("\n" + "=" * 80)
    print(f"Collecting: {repo_name}")
    print("=" * 80)

    try:

        repo = g.get_repo(repo_name)

        data = collect_pr_data(
            repo,
            state="closed",
            max_prs=100
        )

        repo_slug = repo.full_name.replace("/", "_")

        output_path = os.path.join(
            output_dir,
            f"{repo_slug}.jsonl"
        )

        with open(output_path, "w", encoding="utf-8") as f:

            for record in data:
                f.write(json.dumps(record, ensure_ascii=False) + "\n")

        print(f"✅ Saved {len(data)} PRs -> {output_path}")

        total_prs += len(data)

    except Exception as e:

        print(f"❌ Failed: {repo_name}")
        print(e)

print("\n" + "=" * 80)
print(f"Finished collecting {total_prs} Pull Requests.")
print(f"Datasets saved in: {output_dir}")
print("=" * 80)


Collecting: django/django
[1/100] PR #21709: Fixed #37236 -- Allowed altering spatial indexes on RasterField.
[2/100] PR #21629: Fixed #27734 -- Made parallel test workers reuse database clones of exited workers.
[3/100] PR #21706: Removed advice to include ticket numbers in tests.
[4/100] PR #21693: Fixed #37238 -- Fixed unintentional fallback to python default for a db_default.
[5/100] PR #18506: Fixes [#25656] Stop displaying links in recent changes when user does not have view permission
[6/100] PR #21699: Fixed #37235 -- Added compatibility for sqlparse 0.5.5.
[7/100] PR #21700: Fixed #37240 -- Fixed simplify_regex() with multiple unnamed groups.
[8/100] PR #21704: Updated asgiref dependency in free-threaded requirements.
[9/100] PR #21703: Fixed #31923 -- Added Cross-Origin-Embedder-Policy and Cross-Origin-Resource-Policy header support.
